# Konsolidierung der Soll-Datenbasis

Dieses Notebook baut aus den CIS-Rohdaten eine bereinigte Soll-Basis und speichert sie als CSV und Pickle. Die Pickle übernehmen wir später ins Analyse-Notebook, ohne die Vorverarbeitung erneut machen zu müssen. Bereinigt wird nach Franks Ausschlusskriterien (Mafi-Touren raus, doppelte Startzeilen zusammenfassen) und durch eigene Normalisierung von Kennzeichen und Fahrernamen. Den Anfang macht die Reduktion auf die relevanten Spalten samt Format- und Zeitraumkorrektur.

## CIS-Rohdaten einlesen

Die cis_daten.xlsx enthält die TMS-Tour- und Stopp-Daten. Wir setzen vor dem Einlesen die Pandas-Anzeigeoptionen so, dass keine Spalten und keine Zeilenbreite abgeschnitten wird.

In [24]:
import pandas as pd
from pathlib import Path
import re

# Anzeigeoptionen, damit DataFrames vollständig sichtbar sind
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", None)

BASE_PATH = Path.cwd().parent / "data" / "raw"
cis_raw = pd.read_excel(BASE_PATH / "cis_daten.xlsx")
cis_raw

,TOURNR,BEZEICHNUNG,BEMERKUNG,HAUPTTOUR,TOURENART,SENDART,STARTDATUM,STARTKENNUNG,STARTNAME1,STARTNAME2,STARTNAME3,STARTLAND,STARTPLZ,STARTORT,STARTORTSTEIL,STARTORTZUSATZ,STARTSTRASSE,STARTINFO,STARTRELATION,STARTERMITTLUNG,STARTUTCABWEICHUNG,ANKUNFTDATUM,ANKUNFTKENNUNG,ANKUNFTNAME1,ANKUNFTNAME2,ANKUNFTNAME3,ANKUNFTLAND,ANKUNFTPLZ,ANKUNFTORT,ANKUNFTORTSTEIL,ANKUNFTORTZUSATZ,ANKUNFTSTRASSE,ANKUNFTINFO,ANKUNFTRELATION,ANKUNFTERMITTLUNG,ANKUNFTUTCABWEICHUNG,ENTFERNUNG,MAUT_KM,MAUT_KM_MANUELL,EMPFSPED,LKWNR,LKW_KENNZ,ANHNR,ANH_KENNZ,FAHRER,FAHRERNAME,BEGLEITER,BEGLEITERNAME,GERAET1,GERAET1_KENNZ,GERAET2,GERAET2_KENNZ,BUCHFLAG,DRUCKSTATUS,FRFUEHRER,FRZAHLER,GEWICHT,VOLUMEN,LADEMETER,WARENWERT,ANZSTUECK,ANZCOLLI,ANZPAL1,ANZPAL2,ANZPAL3,SOLL_ZEIT,ANZSENDUNGEN,WB1,WB2,SONST1,SONST2,FF_PROJEKT,FF_PROJEKTNR,FF_NR,FF_FAKTNR,FF_MATCHCODE,FF_PREISBILD,FF_DRUCKFLAG,FF_TARIF,FF_MARGE,FF_PREISTARIF,FF_PREISVEREINB,FF_ZEIT,FF_TEXTFLAG,FF_ENTFERNUNG,FF_WC,FZ_PROJEKT,FZ_PROJEKTNR,FZ_NR,FZ_FAKTNR,FZ_MATCHCODE,FZ_PREISBILD,FZ_DRUCKFLAG,FZ_TARIF,FZ_MARGE,FZ_PREISTARIF,FZ_PREISVEREINB,FZ_ZEIT,FZ_TEXTFLAG,FZ_ENTFERNUNG,FZ_WC,GRENZELAND,GRENZEPLZ,GRENZEORT,NL,DISPO_GRP,FZG_GRP,BEARBSTATUS,ANFORDERUNG_FS,ANFORDERUNG_ADR,ANFORDERUNG_SONST,ANFORDERUNG_KFZ,PRIOR_PLAN,PRIOR_EINSATZ,TELEMATIK_ID,TELEMATIK_EXPORT_DATUM,STATIONSSTATUS,KALKULIERTE_KOSTEN,ERLOES,LEERTOUR,EXPFLAG,PLANTOUR,KMSTART,KMANKUNFT,FELDEIGENSCHAFTEN,BEARBEITUNGSKENNZEICHEN,LSTELLPLATZ_ID,ERFASSDATUM,ERFASSNUTZER,AENDERDATUM,AENDERNUTZER,TATS_GEWICHT,STELLPLAETZE,KALKULATIONSART,KALKULATIONSLEVEL,SPERRESTATUS,EMISSION_CO2,ANZSENDPOS,TATS_GEWICHT_MANUELL,VOLUMEN_MANUELL,LADEMETER_MANUELL,STELLPLAETZE_MANUELL,MAUT_KOSTEN_MANUELL,EXTERNE_NUMMER,TELEMATIKWORKFLOWID,NUMMER,TOUR,TOURSTATION_ID,TYP,TOURSTATIONAKTION_ID,ANKUNFT,ABFAHRT,UTCABWEICHUNG,ENTFERNUNG_1,DIFFERENZENTFERNUNG,ADRESSE,NAME1,NAME2,LAND,PLZ,ORT,ORTSTEIL,STRASSE,GEOX,GEOY,GEOSTATUS,TATS_ANKUNFT,TATS_ABFAHRT,ISTZEITSTATUS,ANKUNFT_SOLLVON,ANKUNFT_SOLLBIS,INFOTEXT,FEST,FLAGS,LADEMETER_1,ANFORDERUNG_FS_1,ANFORDERUNG_ADR_1,ANFORDERUNG_SONST_1,ANFORDERUNG_KFZ_1,ERFASSDATUM_1,ERFASSNUTZER_1,AENDERDATUM_1,AENDERNUTZER_1
0,H/42374,Greif,NaN,NaN,0,NaN,02.03.2026 06:00,20457-GreifMafi,Greif Packaging Germany GmbH,NaN,NaN,D,20457,Hamburg,NaN,NaN,Brandenburger Str. 12,NaN,NaN,0,1,02.03.2026 09:23,20457-GreifMafi,Greif Packaging Germany GmbH,NaN,NaN,D,20457,Hamburg,NaN,NaN,Brandenburger Str. 12,NaN,NaN,0,1,NaN,8.4,NaN,NaN,92564-14,Plo-TS856,NaN,NaN,317.0,"Hesse, Sven-Erik",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22926-htsahre,NaN,6480.0,128.6604,0,0,450,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,-2,NaN,NaN,NaN,0.0,NaN,NaN,35.83,1,NaN,NaN,NaN,0,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,15.72,NaN,NaN,NaN,NaN,1,Greif,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,129,0,400.94,1.0,0,NaN,NaN,NaN,NaN,0.0,NaN,26.02.2026 14:32,FS,02.03.2026 17:55,cisfleetsrv,6480.0,0.0,0,0,NaN,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,H/42374,553624,1,NaN,02.03.2026 06:00,02.03.2026 06:00,1,0.00,0.00,20457-GreifMafi,Greif Packaging Germany GmbH,NaN,D,20457.0,Hamburg,NaN,Brandenburger Str. 12,9.98632,53.52348,2.0,02.03.2026 06:59,02.03.2026 06:59,10,NaN,NaN,NaN,1.0,0.0,0.0,NaN,NaN,NaN,NaN,01.03.2026 11:16,FS,01.03.2026 11:16,FS
1,H/42374,Greif,NaN,NaN,0,NaN,02.03.2026 06:00,20457-GreifMafi,Greif Packaging Germany GmbH,NaN,NaN,D,20457,Hamburg,NaN,NaN,Brandenburger Str. 12,NaN,NaN,0,1,02.03.2026 09:23,20457-GreifMafi,Greif Packaging Germany GmbH,NaN,NaN,D,20457,Hamburg,NaN,NaN,Brandenburger Str. 12,NaN,NaN,0,1,NaN,8.4,NaN,NaN,92564-14,Plo-TS856,NaN,NaN,317.0,"Hesse, Sven-Erik",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22926-htsahre,NaN,6480.0,128.6604,0,0,450,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,-2,NaN,NaN,NaN,0.0,NaN,NaN,35.83,1,NaN,NaN,NaN,0,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,15.72,NaN,NaN,NaN,NaN,1,Greif,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,129,0,400.94,1.0,0,NaN,NaN,NaN,NaN,0.0,NaN,26.02.2026 14:32,FS,02.03.2026 17:55,cisfleetsrv,6480.0,0.0,0,0,NaN,0,4,NaN,NaN,NaN,NaN,NaN,Na

Die Cis-Datei enthält 183 Spalten, von denen manche Spalten leer oder irrelevant für die Soll-Daten-Basis sind. Manche Spalten beschreiben sogar Ist-Daten, die laut Frank von SPEDION zurückgeschrieben werden und keinen Mehrwert im Vergleich zu EMRs und PRC-Daten enthalten. Daher werden vermeintliche Ist-Daten aus der Cis-Datei im weiteren Verlauf des Projekts nicht berücksichtigt.

## Spaltenreduktion auf die Soll-Basis

Von den 183 CIS-Spalten tragen die meisten nichts zur Soll-Basis bei, viele sind entweder leer oder konstant. Andere sind reine Ist-Felder (TATS_*, ISTZEITSTATUS, DIFFERENZENTFERNUNG), die SPEDION ins CIS zurückschreibt. Diese Ist-Felder werden bewusst weggelassen, weil die Ist-Daten aus EMR und PRC kommen und die Quellen getrennt bleiben sollen. Übrig bleiben 16 Spalten: Identifikation (TOURNR, NUMMER, TOURSTATION_ID, TYP), Klassifikation (BEZEICHNUNG, TOURENART), Soll-Zeiten (STARTDATUM, ANKUNFT, ABFAHRT), Adresse und Geo (NAME1, ORT, STRASSE, GEOX, GEOY) sowie LKW_KENNZ und FAHRERNAME. TYP und TOURENART werden sicherheitshalber auch mitgenommen, obwohl die Bedeutung ihrer Codewerte aus den Daten nicht belegbar ist.

In [25]:
soll_spalten = [
    "TOURNR", "NUMMER", "TOURSTATION_ID", "TYP", "BEZEICHNUNG", "TOURENART", "STARTDATUM", 
    "ANKUNFT", "ABFAHRT", "NAME1", "ORT", "STRASSE", "GEOX", "GEOY", "LKW_KENNZ", "FAHRERNAME",
]
df_solldaten_cis = cis_raw[soll_spalten].copy()
df_solldaten_cis

,TOURNR,NUMMER,TOURSTATION_ID,TYP,BEZEICHNUNG,TOURENART,STARTDATUM,ANKUNFT,ABFAHRT,NAME1,ORT,STRASSE,GEOX,GEOY,LKW_KENNZ,FAHRERNAME
0,H/42374,1,553624,1,Greif,0,02.03.2026 06:00,02.03.2026 06:00,02.03.2026 06:00,Greif Packaging Germany GmbH,Hamburg,Brandenburger Str. 12,9.98632,53.52348,Plo-TS856,"Hesse, Sven-Erik"
1,H/42374,2,553625,4,Greif,0,02.03.2026 06:00,02.03.2026 06:00,02.03.2026 06:30,Greif Packaging Germany GmbH,Hamburg,Brandenburger Straße 12,9.98632,53.52348,Plo-TS856,"Hesse, Sven-Erik"
2,H/42374,3,553626,8,Greif,0,02.03.2026 06:00,02.03.2026 06:49,02.03.2026 07:19,Shell Deutschland GmbH,Hamburg,Worthdamm 32 Building: Grasbrook Lubrica,9.99383,53.54751,Plo-TS856,"Hesse, Sven-Erik"
3,H/42374,4,553634,4,Greif,0,02.03.2026 06:00,02.03.2026 07:41,02.03.2026 08:11,Greif Packaging Germany GmbH,Hamburg,Brandenburger Str. 12,9.98632,53.52348,Plo-TS856,"Hesse, Sven-Erik"
4,H/42374,5,553635,8,Greif,0,02.03.2026 06:00,02.03.2026 08:31,02.03.2026 09:01,Shell Deutschland GmbH,Hamburg,Worthdamm 32 Building: Grasbrook Lubrica,9.99383,53.54751,Plo-TS856,"Hesse, Sven-Erik"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2795,H/42758,1,557896,5,Greif,0,07.04.2026 06:00,07.04.2026 06:00,07.04.2026 06:00,Greif Packaging Germany GmbH,Hamburg,Brandenburger Str. 12,9.98632,53.52348,PI-EN 444,"Teme, Abdoulaye"
2796,H/42758,2,557897,72,Greif,0,07.04.2026 06:00,07.04.2026 06:57,07.04.2026 07:27,Lubrizol Deutschland GmbH,Hamburg,Billbrookdeich 157,10.09782,53.53610,PI-EN 444,"Teme, Abdoulaye"
2797,H/42758,3,557899,68,Greif,0,07.04.2026 06:00,07.04.2026 07:53,07.04.2026 08:23,Greif Packaging Germany GmbH,Hamburg,Brandenburger Str. 12,9.98632,53.52348,PI-EN 444,"Teme, Abdoulaye"
2798,H/42758,4,557900,72,Greif,0,07.04.2026 06:00,07.04.2026 09:52,07.04.2026 10:22,Blue Cube Germany GmbH Stade,Stade,Buetzflethersand,9.51017,53.64711,PI-EN 444,"Teme, Abdoulaye"


## Untersuchung der reduzierten Soll-Daten

Mit .info() verschaffen wir uns einen Überblick über Datentypen und Befüllung der 16 Spalten. Dabei fällt auf, dass die drei Zeitfelder noch als Text vorliegen, was wir direkt im Anschluss korrigieren.

In [26]:
df_solldaten_cis.info()

<class 'pandas.DataFrame'>
RangeIndex: 2800 entries, 0 to 2799
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   TOURNR          2800 non-null   str    
 1   NUMMER          2800 non-null   int64  
 2   TOURSTATION_ID  2800 non-null   int64  
 3   TYP             2800 non-null   int64  
 4   BEZEICHNUNG     2800 non-null   str    
 5   TOURENART       2800 non-null   int64  
 6   STARTDATUM      2800 non-null   str    
 7   ANKUNFT         2800 non-null   str    
 8   ABFAHRT         2800 non-null   str    
 9   NAME1           2742 non-null   str    
 10  ORT             2742 non-null   str    
 11  STRASSE         2742 non-null   str    
 12  GEOX            2800 non-null   float64
 13  GEOY            2800 non-null   float64
 14  LKW_KENNZ       2800 non-null   str    
 15  FAHRERNAME      2797 non-null   str    
dtypes: float64(2), int64(4), str(10)
memory usage: 720.3 KB


In [27]:
# Zeitstempel von Text in echte datetime-Werte umwandeln
for spalte in ["STARTDATUM", "ANKUNFT", "ABFAHRT"]:
    df_solldaten_cis[spalte] = pd.to_datetime(df_solldaten_cis[spalte], format="%d.%m.%Y %H:%M")

df_solldaten_cis.dtypes

TOURNR                       str
NUMMER                     int64
TOURSTATION_ID             int64
TYP                        int64
BEZEICHNUNG                  str
TOURENART                  int64
STARTDATUM        datetime64[us]
ANKUNFT           datetime64[us]
ABFAHRT           datetime64[us]
NAME1                        str
ORT                          str
STRASSE                      str
GEOX                     float64
GEOY                     float64
LKW_KENNZ                    str
FAHRERNAME                   str
dtype: object

## Auf den Analysezeitraum März beschränken

Die CIS-Daten reichen bis in die erste Aprilwoche, EMR und PRC liegen dagegen nur für März vor. Wir filtern das Soll mit denselben Grenzen auf März, damit alle drei Quellen denselben Zeitraum abdecken. Sonst tauchten die April-Touren später als „ungefahren" auf, obwohl es für sie schlicht keine Ist-Daten gibt. Keine Tour läuft über den Monatswechsel, der Filter trennt also sauber.

In [28]:
# Auf den gemeinsamen Analysezeitraum März beschränken 
vorher = len(df_solldaten_cis)
df_solldaten_cis = df_solldaten_cis[
    (df_solldaten_cis["ANKUNFT"] >= "2026-03-01") &
    (df_solldaten_cis["ANKUNFT"] <  "2026-04-01")
].copy()
print(f"Außerhalb März entfernt: {vorher - len(df_solldaten_cis)} Zeilen | verbleibend: {len(df_solldaten_cis)}")

Außerhalb März entfernt: 325 Zeilen | verbleibend: 2475


## Untersuchung der Kategorien und Entfernen der Mafi- und Fremdeinträge

.nunique() zeigt je Spalte die Anzahl verschiedener Werte. Das trennt die hochkardinalen Identifikatoren wie Touren und Stationen von den wenigen Kategorie- und Code-Spalten. Diese wenigen sehen wir uns als Nächstes einzeln an.

In [29]:
df_solldaten_cis.nunique()

TOURNR             232
NUMMER              77
TOURSTATION_ID    2385
TYP                 16
BEZEICHNUNG          4
TOURENART            3
STARTDATUM          56
ANKUNFT           1925
ABFAHRT           1978
NAME1              188
ORT                 64
STRASSE            252
GEOX               225
GEOY               226
LKW_KENNZ           15
FAHRERNAME          16
dtype: int64

Für die niedrigkardinalen Spalten lesen wir die konkrete Verteilung. In BEZEICHNUNG steht neben Greif und HTH auch Mafi, das laut Frank nicht in die Soll-Basis gehört und im nächsten Schritt rausfliegt. Die Codes in TOURENART und TYP sind aus den Daten nicht ableitbar und bleiben offen.

In [30]:
# Verteilung der niedrigkardinalen Spalten
for spalte in ["BEZEICHNUNG", "TOURENART", "TYP"]:
    display(df_solldaten_cis[spalte].value_counts(dropna=False))

BEZEICHNUNG
Greif    1025
HTH       724
Mafi      688
1          38
Name: count, dtype: int64

TOURENART
0    2440
2      27
1       8
Name: count, dtype: int64

TYP
8      852
4      518
72     314
68     233
2      186
1      128
5      102
10      45
32      43
12      31
76      17
16       2
42       1
17       1
104      1
69       1
Name: count, dtype: int64

## Mafi-Touren und Fremdeinträge entfernen

Die Mafi-Zeilen und die kleine Gruppe mit BEZEICHNUNG 1 tragen alle dasselbe Kennzeichen Mafi Greif. Es handelt sich also um dasselbe Fahrzeug, nur unterschiedlich beschriftet. Dazu kommen drei Zeilen mit Leihwagen statt eines echten Kennzeichens. Deshalb wird nicht über BEZEICHNUNG gefiltert, sondern über LKW_KENNZ.

In [31]:
vorher = len(df_solldaten_cis)
df_solldaten_cis = df_solldaten_cis[
    ~df_solldaten_cis["LKW_KENNZ"].isin(["Mafi Greif", "Leihwagen"])
].copy()
print(f"Entfernt: {vorher - len(df_solldaten_cis)} Zeilen | verbleibend: {len(df_solldaten_cis)}")

Entfernt: 729 Zeilen | verbleibend: 1746


Als Nächstes die Verteilung der Ressourcen-Spalten. Bei den Kennzeichen fällt auf, dass dasselbe Präfix uneinheitlich geschrieben wird (Plo- neben Plö-) und die Formate auseinanderlaufen (mal mit Leerzeichen wie PI-EN 444, mal ohne wie OD-TS8048). Bei den Fahrernamen führt nur Krüger einen Umlaut, den wir der Einheitlichkeit halber ausschreiben. 

In [32]:
# Verteilung der Ressourcen-Spalten
for spalte in ["LKW_KENNZ", "FAHRERNAME"]:
    display(df_solldaten_cis[spalte].value_counts(dropna=False))

LKW_KENNZ
PI-EN 444    262
Plo-TS859    218
Plö-TS853    212
OD-TS8041    151
OD-TS8050    133
OD-TS8048    126
Plo-TS856    122
OD-TS8044    121
WL-PL431     116
OD-TS8046     93
PI-EN 900     81
Plö-TS857     64
RW435         47
Name: count, dtype: int64

FAHRERNAME
Giewon, Mariusz             262
Krüger, Alexander           222
Borrs, Thomas               197
Ledwina, Jens               193
Jorczik, Christian          158
Szmajdewicz, Marcin         132
Oelschlaeger, Marcel        130
Hesse, Sven-Erik            122
Calina, Florin Alexandru    105
Hrzuska, Krzystof            83
Teme, Abdoulaye              72
Staben, Volker               47
Schulz, Frank                12
Sowa, Mariusz                11
Name: count, dtype: int64

## Fahrernamen normalisieren
Umlaute schreiben wir einheitlich aus (ä -> ae, ö -> oe, ü -> ue, ß -> ss). Praktisch betrifft das nur Krüger -> Krueger, alle anderen Namen enthalten keine Umlaute.

In [33]:
umlaut_expand = str.maketrans(
    {"ä": "ae", "ö": "oe", "ü": "ue", "Ä": "Ae", "Ö": "Oe", "Ü": "Ue", "ß": "ss"}
)
df_solldaten_cis["FAHRERNAME"] = df_solldaten_cis["FAHRERNAME"].str.translate(umlaut_expand)

print(df_solldaten_cis["FAHRERNAME"].value_counts(dropna=False))

FAHRERNAME
Giewon, Mariusz             262
Krueger, Alexander          222
Borrs, Thomas               197
Ledwina, Jens               193
Jorczik, Christian          158
Szmajdewicz, Marcin         132
Oelschlaeger, Marcel        130
Hesse, Sven-Erik            122
Calina, Florin Alexandru    105
Hrzuska, Krzystof            83
Teme, Abdoulaye              72
Staben, Volker               47
Schulz, Frank                12
Sowa, Mariusz                11
Name: count, dtype: int64


## Kennzeichen normalisieren
Die Kennzeichen werden auf ein einheitliches Muster gebracht. Großschreibung, Umlaute zusammengezogen (Plö -> PLO) und Buchstaben- sowie Zifferngruppen per Bindestrich getrennt (OD-TS8048 -> OD-TS-8048, PI-EN 444 -> PI-EN-444). Anders als bei den Fahrernamen ziehen wir die Umlaute hier zusammen, statt sie auszuschreiben.

In [34]:
umlaut_collapse = str.maketrans({"Ä": "A", "Ö": "O", "Ü": "U"})

def normalisiere_kennzeichen(wert):
    if pd.isna(wert):
        return wert
    wert = wert.upper().translate(umlaut_collapse)
    return "-".join(re.findall(r"[A-Z]+|[0-9]+", wert))   # Buchstaben- und Zifferngruppen trennen

df_solldaten_cis["LKW_KENNZ"] = df_solldaten_cis["LKW_KENNZ"].map(normalisiere_kennzeichen)
display(df_solldaten_cis["LKW_KENNZ"].value_counts(dropna=False))

LKW_KENNZ
PI-EN-444     262
PLO-TS-859    218
PLO-TS-853    212
OD-TS-8041    151
OD-TS-8050    133
OD-TS-8048    126
PLO-TS-856    122
OD-TS-8044    121
WL-PL-431     116
OD-TS-8046     93
PI-EN-900      81
PLO-TS-857     64
RW-435         47
Name: count, dtype: int64

## Fahrzeug ohne Telematik entfernen

RW-435 taucht zwar in den Plandaten auf, hat aber weder in EMR noch in PRC Telematikdaten. Seine geplanten Touren ließen sich nie mit einer Ausführung vergleichen, deshalb nehmen wir es aus der Soll-Basis.

In [35]:
# RW-435 hat in keiner Ist-Quelle (EMR, PRC) Telematik
vorher = len(df_solldaten_cis)
df_solldaten_cis = df_solldaten_cis[df_solldaten_cis["LKW_KENNZ"] != "RW-435"].copy()
print(f"RW-435 entfernt: {vorher - len(df_solldaten_cis)} Zeilen | verbleibend: {len(df_solldaten_cis)}")

RW-435 entfernt: 47 Zeilen | verbleibend: 1699


Zur Kontrolle die Verteilung nach der Normalisierung. Wichtig ist, dass sich die Zahl verschiedener Kennzeichen und Fahrer nur dort ändert, wo wir es erklären können. Die Vereinheitlichung soll Schreibweisen zusammenführen, nicht echte Fahrzeuge oder Fahrer verschmelzen. Von 16 Fahrern bleiben 13: Mirzaie fuhr ausschließlich Mafi und fällt mit dem Mafi-Filter weg, Staben fuhr ausschließlich RW-435 und fällt mit dessen Entfernung weg. Stoffer fällt zuletzt durch den März-Filter, weil seine verbliebenen Touren alle im April liegen. Fehlende Fahrernamen gibt es danach keine mehr, denn die leeren Einträge steckten ebenfalls in den entfernten Mafi- und Leihwagen-Zeilen.

In [36]:
# Kontrolle ob die Kennzeichen einheitlich formatiert sind
display(df_solldaten_cis["LKW_KENNZ"].value_counts(dropna=False))
display(df_solldaten_cis["FAHRERNAME"].value_counts(dropna=False))

LKW_KENNZ
PI-EN-444     262
PLO-TS-859    218
PLO-TS-853    212
OD-TS-8041    151
OD-TS-8050    133
OD-TS-8048    126
PLO-TS-856    122
OD-TS-8044    121
WL-PL-431     116
OD-TS-8046     93
PI-EN-900      81
PLO-TS-857     64
Name: count, dtype: int64

FAHRERNAME
Giewon, Mariusz             262
Krueger, Alexander          222
Borrs, Thomas               197
Ledwina, Jens               193
Jorczik, Christian          158
Szmajdewicz, Marcin         132
Oelschlaeger, Marcel        130
Hesse, Sven-Erik            122
Calina, Florin Alexandru    105
Hrzuska, Krzystof            83
Teme, Abdoulaye              72
Schulz, Frank                12
Sowa, Mariusz                11
Name: count, dtype: int64

## Zwischenkontrolle vor den letzten beiden Schritten

An dieser Stelle stehen 232 Touren auf 1934 Zeilen, im Schnitt rund acht Stationen je Tour (zwischen 3 und 20). Zwei Doppelungen sind bislang ungeprüft, erst danach steht die endgültige Vergleichsbasis fest. Wir sehen uns als Erstes die fehlenden Werte an, falls Zeilen unvollständig sind oder nachgefüllt werden müssen.

In [37]:
print("Touren:", df_solldaten_cis["TOURNR"].nunique())
df_solldaten_cis.groupby("TOURNR").size().describe()

Touren: 203


count    203.000000
mean       8.369458
std        3.652810
min        3.000000
25%        5.000000
50%        8.000000
75%       11.000000
max       20.000000
dtype: float64

TOURSTATION_ID ist nicht eindeutig, einzelne IDs wiederholen sich über aufeinanderfolgende NUMMER innerhalb einer Tour. Das halten wir fest, bewerten aber noch nicht, wofür die Wiederholung steht, da sich das aus den Daten allein nicht klären lässt.

In [38]:
dubletten = df_solldaten_cis[df_solldaten_cis["TOURSTATION_ID"].duplicated(keep=False)]
print(f"Doppelte TOURSTATION_ID: {df_solldaten_cis['TOURSTATION_ID'].duplicated().sum()} | betroffene Zeilen: {len(dubletten)}")
dubletten.sort_values(["TOURSTATION_ID", "NUMMER"])[["TOURNR", "NUMMER", "TOURSTATION_ID", "NAME1", "ORT"]].head(15)

Doppelte TOURSTATION_ID: 17 | betroffene Zeilen: 29


,TOURNR,NUMMER,TOURSTATION_ID,NAME1,ORT
707,H/42445,3,554343,Greif Packaging Germany GmbH,Hamburg
708,H/42445,4,554343,Greif Packaging Germany GmbH,Hamburg
792,H/42459,3,554528,NaN,NaN
793,H/42459,4,554528,Greif Packaging Germany GmbH,Hamburg
794,H/42459,5,554529,Hywax GmbH,Hamburg
795,H/42459,6,554529,NaN,NaN
796,H/42459,7,554529,Hywax GmbH,Hamburg
797,H/42459,8,554530,Greif Packaging Germany GmbH,Hamburg
798,H/42459,9,554530,NaN,NaN
799,H/42459,10,554530,Greif Packaging Germany GmbH,Hamburg


Als nächstes nochmal einen Überblick über fehlende Werte verschaffen.

In [39]:
df_solldaten_cis.isna().sum()

TOURNR             0
NUMMER             0
TOURSTATION_ID     0
TYP                0
BEZEICHNUNG        0
TOURENART          0
STARTDATUM         0
ANKUNFT            0
ABFAHRT            0
NAME1             10
ORT               10
STRASSE           10
GEOX               0
GEOY               0
LKW_KENNZ          0
FAHRERNAME         0
dtype: int64

Bei den fehlenden Werten zeigt sich, dass nur NAME1, ORT und STRASSE betroffen sind, jeweils in zehn Zeilen. Alles andere ist vollständig, die zehn adresslosen Zeilen sehen wir uns einzeln an.

In [40]:
df_solldaten_cis[df_solldaten_cis["NAME1"].isna()]

,TOURNR,NUMMER,TOURSTATION_ID,TYP,BEZEICHNUNG,TOURENART,STARTDATUM,ANKUNFT,ABFAHRT,NAME1,ORT,STRASSE,GEOX,GEOY,LKW_KENNZ,FAHRERNAME
792,H/42459,3,554528,4,Greif,0,2026-03-10 06:00:00,2026-03-10 07:00:00,2026-03-10 07:30:00,NaN,NaN,NaN,0.0,0.0,PLO-TS-859,"Krueger, Alexander"
795,H/42459,6,554529,8,Greif,0,2026-03-10 06:00:00,2026-03-10 08:30:00,2026-03-10 09:00:00,NaN,NaN,NaN,0.0,0.0,PLO-TS-859,"Krueger, Alexander"
798,H/42459,9,554530,4,Greif,0,2026-03-10 06:00:00,2026-03-10 10:00:00,2026-03-10 10:30:00,NaN,NaN,NaN,0.0,0.0,PLO-TS-859,"Krueger, Alexander"
800,H/42459,11,554531,8,Greif,0,2026-03-10 06:00:00,2026-03-10 11:00:00,2026-03-10 11:30:00,NaN,NaN,NaN,0.0,0.0,PLO-TS-859,"Krueger, Alexander"
1194,H/42510,3,555476,8,Greif,0,2026-03-16 06:00:00,2026-03-16 07:00:00,2026-03-16 07:30:00,NaN,NaN,NaN,0.0,0.0,PLO-TS-859,"Krueger, Alexander"
1196,H/42510,5,555440,4,Greif,0,2026-03-16 06:00:00,2026-03-16 08:00:00,2026-03-16 08:30:00,NaN,NaN,NaN,0.0,0.0,PLO-TS-859,"Krueger, Alexander"
1199,H/42510,8,555477,8,Greif,0,2026-03-16 06:00:00,2026-03-16 09:30:00,2026-03-16 10:00:00,NaN,NaN,NaN,0.0,0.0,PLO-TS-859,"Krueger, Alexander"
1201,H/42510,10,555471,4,Greif,0,2026-03-16 06:00:00,2026-03-16 10:30:00,2026-03-16 11:00:00,NaN,NaN,NaN,0.0,0.0,PLO-TS-859,"Krueger, Alexander"
1205,H/42510,14,555473,4,Greif,0,2026-03-16 06:00:00,2026-03-16 12:30:00,2026-03-16 13:00:00,NaN,NaN,NaN,0.0,0.0,PLO-TS-859,"Krueger, Alexander"
1207,H/42510,16,555472,8,Greif,0,2026-03-16 06:00:00,2026-03-16 13:30:00,2026-03-16 14:00:00,NaN,NaN,NaN,0.0,0.0,PLO-TS-859,"Krueger, Alexander"


Die betroffenen Zeilen stammen aus nur zwei Touren, haben keine Adresse und entsprechend Koordinaten von 0, 0, aber dafür gültige Reihenfolge und Soll-Zeiten. Bevor wir über das Löschen entscheiden, prüfen wir, ob dieselbe Station an anderer Stelle mit Adresse auftaucht.

In [41]:
miss_ids = df_solldaten_cis.loc[df_solldaten_cis["NAME1"].isna(), "TOURSTATION_ID"].unique()
df_solldaten_cis[df_solldaten_cis["TOURSTATION_ID"].isin(miss_ids)] \
    .sort_values(["TOURSTATION_ID", "NUMMER"]) \
    [["TOURNR", "NUMMER", "TOURSTATION_ID", "TYP", "NAME1", "ORT", "GEOX", "GEOY"]]

,TOURNR,NUMMER,TOURSTATION_ID,TYP,NAME1,ORT,GEOX,GEOY
792,H/42459,3,554528,4,NaN,NaN,0.00000,0.00000
793,H/42459,4,554528,68,Greif Packaging Germany GmbH,Hamburg,9.98632,53.52348
794,H/42459,5,554529,72,Hywax GmbH,Hamburg,9.98213,53.53497
795,H/42459,6,554529,8,NaN,NaN,0.00000,0.00000
796,H/42459,7,554529,72,Hywax GmbH,Hamburg,9.98213,53.53497
797,H/42459,8,554530,68,Greif Packaging Germany GmbH,Hamburg,9.98632,53.52348
798,H/42459,9,554530,4,NaN,NaN,0.00000,0.00000
799,H/42459,10,554530,68,Greif Packaging Germany GmbH,Hamburg,9.98632,53.52348
800,H/42459,11,554531,8,NaN,NaN,0.00000,0.00000
801,H/42459,12,554531,72,Hywax GmbH,Hamburg,9.98213,53.53497


Tatsächlich kommt jede dieser TOURSTATION_IDs in derselben Tour ein weiteres Mal vor, dort mit vollständiger Adresse und gültigen Koordinaten. Der Ort ist also bekannt und steht bereits in der Datei. Statt zu löschen, füllen wir die leeren Felder aus dieser Zwillingszeile.

## Adressen und Koordinaten nachfüllen
Je TOURSTATION_ID sind Adresse und Koordinaten eindeutig. Aus den befüllten Zeilen bauen wir eine Referenz und übertragen NAME1, PLZ, ORT, STRASSE, GEOX und GEOY auf die leeren Zeilen. Danach gibt es weder fehlende Adressen noch Null-Koordinaten.

In [42]:
fill_cols = ["NAME1", "ORT", "STRASSE", "GEOX", "GEOY"]

# Referenz je TOURSTATION_ID aus den Zeilen mit Adresse
referenz = (
    df_solldaten_cis[df_solldaten_cis["NAME1"].notna()]
    .drop_duplicates("TOURSTATION_ID")
    .set_index("TOURSTATION_ID")[fill_cols]
)

maske = df_solldaten_cis["NAME1"].isna()
for spalte in fill_cols:
    df_solldaten_cis.loc[maske, spalte] = df_solldaten_cis.loc[maske, "TOURSTATION_ID"].map(referenz[spalte])

print("NAME1 fehlen:", df_solldaten_cis["NAME1"].isna().sum())
print("Null-Koordinaten:", ((df_solldaten_cis["GEOX"] == 0) & (df_solldaten_cis["GEOY"] == 0)).sum())

NAME1 fehlen: 0
Null-Koordinaten: 0


In [43]:
# Gibt es je (Tour, Ort, Ankunft) mehr als eine Zeile?
dop_maske = df_solldaten_cis.duplicated(["TOURNR", "GEOX", "GEOY", "ANKUNFT"], keep=False)
print(f"Zeilen in Doppelungs-Gruppen: {dop_maske.sum()}")
df_solldaten_cis[dop_maske].sort_values(["TOURNR", "ANKUNFT", "ABFAHRT"])[
    ["TOURNR", "NUMMER", "TYP", "NAME1", "ANKUNFT", "ABFAHRT"]
].head(8)

Zeilen in Doppelungs-Gruppen: 244


,TOURNR,NUMMER,TYP,NAME1,ANKUNFT,ABFAHRT
0,H/42374,1,1,Greif Packaging Germany GmbH,2026-03-02 06:00:00,2026-03-02 06:00:00
1,H/42374,2,4,Greif Packaging Germany GmbH,2026-03-02 06:00:00,2026-03-02 06:30:00
13,H/42376,1,1,Greif Packaging Germany GmbH,2026-03-02 06:00:00,2026-03-02 06:00:00
14,H/42376,2,4,Greif Packaging Germany GmbH,2026-03-02 06:00:00,2026-03-02 06:30:00
35,H/42379,1,1,HTH Hanse GmbH,2026-03-02 05:00:00,2026-03-02 05:00:00
36,H/42379,2,4,HTH Hanse GmbH,2026-03-02 05:00:00,2026-03-02 06:45:00
45,H/42380,1,1,Greif Packaging Germany GmbH,2026-03-02 06:00:00,2026-03-02 06:00:00
46,H/42380,2,4,Greif Packaging Germany GmbH,2026-03-02 06:00:00,2026-03-02 06:30:00


## Doppelte Startzeilen zusammenfassen

Das Planungssystem legt manche Stopps doppelt an, das heißt gleiche Tour, gleicher Ort, gleiche geplante Ankunft, aber zwei Zeilen mit unterschiedlichem TYP (typisch TYP 1 und TYP 4) und
unterschiedlicher Abfahrt. Frank hat bestätigt, dass dies eine Systemdoppelung ist und als eine Zeile zu werten ist. Eine manuelle Überprüfung mit den PRC-Daten zeigt, dass PRC ohnehin nur eine derbeiden Zeilen meldet und zwar die mit der späteren Abfahrt. Also behalten wir je Gruppe genau diese Zeile und lassen NUMMER unverändert, damit die Stopp-Nummer weiterhin der PRC-Stoppkennung
entspricht. Diese Doppelung ist nicht zu verwechseln mit der separaten TOURSTATION_ID-Wiederholung weiter oben, bei der dieselbe Station zu verschiedenen Ankunftszeiten geplant ist. Die bleiben erstmal erhalten, weil ihre Ankunftszeiten sich unterscheiden.

In [44]:
vorher = len(df_solldaten_cis)
df_solldaten_cis = (
    df_solldaten_cis
    .sort_values(["TOURNR", "GEOX", "GEOY", "ANKUNFT", "ABFAHRT"])  
    .drop_duplicates(subset=["TOURNR", "GEOX", "GEOY", "ANKUNFT"], keep="last")
    .sort_values(["TOURNR", "NUMMER"])
    .reset_index(drop=True)
)
print(f"Doppelte Startzeilen entfernt: {vorher - len(df_solldaten_cis)} | verbleibend: {len(df_solldaten_cis)}")

Doppelte Startzeilen entfernt: 122 | verbleibend: 1577


## Soll-Basis abschließen und exportieren

Die bereinigte Soll-Basis speichern wir in zwei Formaten. Das Pickle ist das Arbeitsformat für die Folge-Notebooks und erhält die Datentypen, sodass die Zeitstempel beim Einlesen direkt wieder als datetime vorliegen und nicht erneut geparst werden müssen. Die CSV ist die lesbare Beleg-Kopie für den Anhang und zum Sichten in Excel. Zur Kontrolle lesen wir das Pickle einmal ein und prüfen die Datentypen.

In [45]:
df_solldaten_cis_clean = df_solldaten_cis.copy()

OUT_PATH = Path.cwd().parent / "data" / "processed"
OUT_PATH.mkdir(parents=True, exist_ok=True)

# Export der bereinigten Soll-Daten für CIS als Pickle und CSV
df_solldaten_cis_clean.to_pickle(OUT_PATH / "solldaten_cis_clean.pkl")
df_solldaten_cis_clean.to_csv(OUT_PATH / "solldaten_cis_clean.csv", index=False, encoding="utf-8-sig")

print("Form:", df_solldaten_cis_clean.shape)
print(df_solldaten_cis_clean.dtypes)

Form: (1577, 16)
TOURNR                       str
NUMMER                     int64
TOURSTATION_ID             int64
TYP                        int64
BEZEICHNUNG                  str
TOURENART                  int64
STARTDATUM        datetime64[us]
ANKUNFT           datetime64[us]
ABFAHRT           datetime64[us]
NAME1                        str
ORT                          str
STRASSE                      str
GEOX                     float64
GEOY                     float64
LKW_KENNZ                    str
FAHRERNAME                   str
dtype: object


In [46]:
df_solldaten_cis_clean.head()

,TOURNR,NUMMER,TOURSTATION_ID,TYP,BEZEICHNUNG,TOURENART,STARTDATUM,ANKUNFT,ABFAHRT,NAME1,ORT,STRASSE,GEOX,GEOY,LKW_KENNZ,FAHRERNAME
0,H/42374,2,553625,4,Greif,0,2026-03-02 06:00:00,2026-03-02 06:00:00,2026-03-02 06:30:00,Greif Packaging Germany GmbH,Hamburg,Brandenburger Straße 12,9.98632,53.52348,PLO-TS-856,"Hesse, Sven-Erik"
1,H/42374,3,553626,8,Greif,0,2026-03-02 06:00:00,2026-03-02 06:49:00,2026-03-02 07:19:00,Shell Deutschland GmbH,Hamburg,Worthdamm 32 Building: Grasbrook Lubrica,9.99383,53.54751,PLO-TS-856,"Hesse, Sven-Erik"
2,H/42374,4,553634,4,Greif,0,2026-03-02 06:00:00,2026-03-02 07:41:00,2026-03-02 08:11:00,Greif Packaging Germany GmbH,Hamburg,Brandenburger Str. 12,9.98632,53.52348,PLO-TS-856,"Hesse, Sven-Erik"
3,H/42374,5,553635,8,Greif,0,2026-03-02 06:00:00,2026-03-02 08:31:00,2026-03-02 09:01:00,Shell Deutschland GmbH,Hamburg,Worthdamm 32 Building: Grasbrook Lubrica,9.99383,53.54751,PLO-TS-856,"Hesse, Sven-Erik"
4,H/42374,6,553627,2,Greif,0,2026-03-02 06:00:00,2026-03-02 09:23:00,2026-03-02 09:23:00,Greif Packaging Germany GmbH,Hamburg,Brandenburger Str. 12,9.98632,53.52348,PLO-TS-856,"Hesse, Sven-Erik"
